# Pemodelan LSTM untuk Prediksi Tinggi Muka Air (TMA) di DAS Peusangan dan Sekitarnya

Notebook ini mengimplementasikan alur lengkap pengolahan data dan pemodelan *Long Short-Term Memory* (LSTM) untuk prediksi Tinggi Muka Air (TMA) sungai berdasarkan rancangan arsitektur Penelitian Magister Teknologi Informasi, Fakultas Teknik, Universitas Malikussaleh:

- **Arsitektur Inti**: `LSTM(128) -> Dropout(0.2) -> LSTM(64) -> Dense(32, ReLU) -> Dense(1, Linear)`
- **Pra-pemrosesan**: Pembersihan data, normalisasi `MinMaxScaler [0, 1]` (di-*fit* hanya pada data latih untuk mencegah *data leakage*), dan transformasi deret waktu menjadi *supervised learning* menggunakan *sliding window*.
- **Optimasi & Loss**: Optimizer `Adam` dan fungsi kerugian `MSE` (*Mean Squared Error*).
- **Metrik Evaluasi**: `RMSE`, `MAE`, $R^2$, `NSE` (*Nash-Sutcliffe Efficiency*), serta akurasi klasifikasi status siaga banjir berbasis ambang persentil empiris.
- **Eksperimen Komparatif**:
  1. Perbandingan skenario input data skala harian (Univariat TMA, Multivariat BWS+ARR, Multivariat 5 Pilar BWS+ERA5-Land, dan Multivariat Spasial Lintas Stasiun).
  2. Interpolasi temporal ke skala per jam (`Linear`, `PCHIP`, `Cubic Spline`) untuk prediksi *lead time* $t+1$, $t+3$, $t+6$, dan $t+7$ jam, termasuk integrasi dengan data ERA5-Land *hourly* aktual.
  3. Studi ablasi variasi jumlah unit LSTM dan *learning rate* optimizer `Adam`.

> **Catatan Sumber Data**: Seluruh berkas pada direktori `../../Data` diakses dalam mode **read-only**. Seluruh artefak hasil pelatihan disimpan ke `./outputs/`.


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reproduksibilitas
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Konfigurasi visualisasi
plt.rcParams["figure.figsize"] = (12, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
sns.set_style("whitegrid")

# Path relatif (Data bersifat read-only)
DATA_DIR = Path("../../Data")
OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("DATA_DIR exists:", DATA_DIR.exists())
print("OUT_DIR:", OUT_DIR.resolve())


## 1. Fungsi Utilitas: Sliding Window, Arsitektur LSTM, & Evaluasi

Seluruh eksperimen menggunakan protokol evaluasi yang seragam:
1. **Temporal Split**: 70% Data Latih (*Train*), 15% Validasi (*Validation*), 15% Uji (*Test*) secara kronologis tanpa pengacakan.
2. **Normalisasi**: `MinMaxScaler(feature_range=(0, 1))` di-*fit* khusus pada `Train` kemudian diterapkan ke `Validation` dan `Test`.
3. **Metrik Hidrologi & ML**:
   - $\text{RMSE} = \sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2}$
   - $\text{NSE} = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$ (nilai mendekati 1 menunjukkan kesesuaian hidrograf yang sangat baik)
   - $R^2$ dan $\text{MAE}$
   - Akurasi Klasifikasi Siaga Banjir (berdasarkan ambang empiris persentil ke-75/Waspada pada data latih).


In [ ]:
def create_sequences(X_data: np.ndarray, y_data: np.ndarray, window_size: int, horizon: int = 1):
    """Membentuk pasangan (X, y) berbasis sliding window untuk prediksi t + horizon."""
    Xs, ys = [], []
    max_idx = len(X_data) - window_size - horizon + 1
    for i in range(max_idx):
        Xs.append(X_data[i : i + window_size])
        ys.append(y_data[i + window_size + horizon - 1])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


def nash_sutcliffe_efficiency(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Menghitung Nash-Sutcliffe Efficiency (NSE)."""
    denom = np.sum((y_true - np.mean(y_true)) ** 2)
    if denom == 0:
        return np.nan
    return float(1.0 - np.sum((y_true - y_pred) ** 2) / denom)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, alert_threshold: float = None) -> dict:
    """Menghitung RMSE, MAE, R2, NSE, dan akurasi ambang siaga."""
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    nse = nash_sutcliffe_efficiency(y_true, y_pred)
    acc_siaga = np.nan
    if alert_threshold is not None:
        true_cls = (y_true >= alert_threshold).astype(int)
        pred_cls = (y_pred >= alert_threshold).astype(int)
        acc_siaga = float(np.mean(true_cls == pred_cls) * 100.0)
    return {
        "RMSE (m)": rmse,
        "MAE (m)": mae,
        "R2": r2,
        "NSE": nse,
        "Alert_Acc (%)": acc_siaga,
    }


def build_lstm_model(
    input_shape: tuple,
    lstm_units: tuple = (128, 64),
    dropout_rate: float = 0.2,
    dense_units: int = 32,
    learning_rate: float = 1e-3,
):
    """Membangun model LSTM sesuai rancangan arsitektur penelitian."""
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = Sequential([
        Input(shape=input_shape),
        LSTM(lstm_units[0], return_sequences=True),
        Dropout(dropout_rate),
        LSTM(lstm_units[1], return_sequences=False),
        Dense(dense_units, activation="relu"),
        Dense(1, activation="linear"),
    ])
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=["mae"],
    )
    return model


def run_lstm_experiment(
    df: pd.DataFrame,
    feature_cols: list,
    target_col: str,
    exp_name: str,
    window_size: int = 14,
    horizon: int = 1,
    lstm_units: tuple = (128, 64),
    dropout_rate: float = 0.2,
    dense_units: int = 32,
    learning_rate: float = 1e-3,
    epochs: int = 50,
    batch_size: int = 32,
    verbose: int = 0,
):
    """Menjalankan pipeline penuh: temporal split, MinMax [0,1], sliding window, training, & evaluasi."""
    data = df[feature_cols + [target_col]].copy().dropna()
    X_raw = df[feature_cols].loc[data.index].values.astype(np.float32)
    y_raw = df[target_col].loc[data.index].values.astype(np.float32).reshape(-1, 1)

    n = len(data)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    # Fit scaler HANYA pada data train
    x_scaler = MinMaxScaler(feature_range=(0, 1))
    y_scaler = MinMaxScaler(feature_range=(0, 1))

    X_train_s = x_scaler.fit_transform(X_raw[:train_end])
    X_val_s = x_scaler.transform(X_raw[train_end:val_end])
    X_test_s = x_scaler.transform(X_raw[val_end:])

    y_train_s = y_scaler.fit_transform(y_raw[:train_end])
    y_val_s = y_scaler.transform(y_raw[train_end:val_end])
    y_test_s = y_scaler.transform(y_raw[val_end:])

    X_train, y_train = create_sequences(X_train_s, y_train_s, window_size, horizon)
    X_val, y_val = create_sequences(X_val_s, y_val_s, window_size, horizon)
    X_test, y_test = create_sequences(X_test_s, y_test_s, window_size, horizon)

    model = build_lstm_model(
        input_shape=(window_size, len(feature_cols)),
        lstm_units=lstm_units,
        dropout_rate=dropout_rate,
        dense_units=dense_units,
        learning_rate=learning_rate,
    )

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=verbose,
    )

    y_pred_s = model.predict(X_test, verbose=0)
    y_test_inv = y_scaler.inverse_transform(y_test).ravel()
    y_pred_inv = y_scaler.inverse_transform(y_pred_s).ravel()

    # Ambang batas Waspada empiris (P75 dari data latih)
    p75_train = float(np.percentile(y_raw[:train_end], 75))
    metrics = compute_metrics(y_test_inv, y_pred_inv, alert_threshold=p75_train)
    metrics.update({
        "Experiment": exp_name,
        "Num_Features": len(feature_cols),
        "Window": window_size,
        "Horizon": horizon,
        "LSTM_Units": f"{lstm_units[0]}-{lstm_units[1]}",
        "Adam_LR": learning_rate,
        "Epochs_Run": len(history.history["loss"]),
        "N_Train": len(X_train),
        "N_Test": len(X_test),
    })

    test_dates = data.index[val_end + window_size + horizon - 1 :]
    return {
        "metrics": metrics,
        "model": model,
        "history": history.history,
        "y_true": y_test_inv,
        "y_pred": y_pred_inv,
        "test_index": test_dates[: len(y_test_inv)],
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
    }

# Store all experiment results
ALL_RESULTS = []
print("Utility functions initialized.")


## 2. Eksplorasi & Pra-pemrosesan Dataset Sumber

Dataset yang digunakan bersumber dari direktori `../../Data` (read-only):
1. **TMA Harian Keureuto (2015–2024)**: `02_Hidrologi_TMA_BWS/02_data_penelitian/tma_keureuto_harian_2015_2024_penelitian.csv`
2. **Curah Hujan Harian ARR Keureuto & Pasangan Hujan–TMA (2017–2020)**: `03_Meteorologi_ARR_Curah_Hujan/02_data_penelitian/arr_keureuto_harian_2017_2020_penelitian.csv` & `pasangan_hujan_tma_keureuto_2017_2020_penelitian.csv`
3. **TMA Sub-Harian Krueng Simpo (2020 — Pagi, Siang, Sore)**: `02_Hidrologi_TMA_BWS/02_data_penelitian/tma_krueng_simpo_2020_penelitian.csv`
4. **TMA Lintas Stasiun WS Pase–Peusangan (2020)**: `02_Hidrologi_TMA_BWS/02_data_penelitian/tma_ws_pase_peusangan_2020_wide_penelitian.csv`
5. **Reanalisis ERA5-Land Daily & Hourly (2021–2025)**: `05_Klimatologi_Reanalysis_ERA5/` (curah hujan, *surface/subsurface runoff*, kelembaban tanah `swvl1`–`swvl2`, tekanan permukaan, suhu, kelembaban relatif)
6. **Telemetri Sensor IoT Aktual & Log Prediksi (2026)**: `13_Pengujian_IoT/terstruktur/04_telemetri_sensor_2618_baris.csv` & `10_prediksi_ai_lstm_2750_baris.csv`.


In [ ]:
# 1. TMA Keureuto Harian (2015-2024)
df_tma_k = pd.read_csv(
    DATA_DIR / "02_Hidrologi_TMA_BWS/02_data_penelitian/tma_keureuto_harian_2015_2024_penelitian.csv",
    parse_dates=["tanggal"],
).sort_values("tanggal").set_index("tanggal")

# 2. ARR Curah Hujan Keureuto & Pasangan Hujan-TMA (2017-2020, struktur terbarukan)
df_arr = pd.read_csv(
    DATA_DIR / "03_Meteorologi_ARR_Curah_Hujan/02_data_penelitian/arr_keureuto_harian_2017_2020_penelitian.csv",
    parse_dates=["tanggal"],
).sort_values("tanggal").set_index("tanggal")

df_pair_arr_tma = pd.read_csv(
    DATA_DIR / "03_Meteorologi_ARR_Curah_Hujan/02_data_penelitian/pasangan_hujan_tma_keureuto_2017_2020_penelitian.csv",
    parse_dates=["tanggal"],
).sort_values("tanggal").set_index("tanggal")

# 3. TMA Krueng Simpo 2020 (Pagi, Siang, Sore)
df_simpo = pd.read_csv(
    DATA_DIR / "02_Hidrologi_TMA_BWS/02_data_penelitian/tma_krueng_simpo_2020_penelitian.csv",
    parse_dates=["tanggal"],
).sort_values("tanggal").set_index("tanggal")

# 4. TMA WS Pase Peusangan Wide 2020
df_ws = pd.read_csv(
    DATA_DIR / "02_Hidrologi_TMA_BWS/02_data_penelitian/tma_ws_pase_peusangan_2020_wide_penelitian.csv",
    parse_dates=["tanggal"],
).sort_values("tanggal").set_index("tanggal")

# 5. ERA5 Daily (Pos_Keureuto) & Hourly
df_era5_daily = pd.read_csv(
    DATA_DIR / "05_Klimatologi_Reanalysis_ERA5/era5_das_peusangan_daily_2021_2025.csv",
    parse_dates=["date"],
)
df_era5_k_daily = (
    df_era5_daily[df_era5_daily["station"] == "Pos_Keureuto"]
    .sort_values("date")
    .set_index("date")
)

df_era5_k_hourly = pd.read_csv(
    DATA_DIR / "05_Klimatologi_Reanalysis_ERA5/Ekstraksi_6_Titik_Independen/hourly_06_krueng_keureuto_awlr2_2021_2025.csv",
    parse_dates=["timestamp_wib"],
).sort_values("timestamp_wib").set_index("timestamp_wib")

# 6. Telemetri Sensor IoT Aktual & Log Prediksi
df_iot_raw = pd.read_csv(
    DATA_DIR / "13_Pengujian_IoT/terstruktur/04_telemetri_sensor_2618_baris.csv",
    parse_dates=["created_at"],
).sort_values("created_at")

df_iot_pred_log = pd.read_csv(
    DATA_DIR / "13_Pengujian_IoT/terstruktur/10_prediksi_ai_lstm_2750_baris.csv",
    parse_dates=["created_at"],
).sort_values("created_at")

print(f"Water Level Keureuto (2015-2024) : {df_tma_k.shape}, missing WL: {df_tma_k['tma_meter_terkoreksi'].isna().sum()}")
print(f"ARR Rainfall Keureuto (2017-2020): {df_arr.shape}, missing RF: {df_arr['curah_hujan_mm'].isna().sum()}")
print(f"Paired Rainfall-WL (2017-2020)   : {df_pair_arr_tma.shape}")
print(f"Water Level Krueng Simpo (2020)  : {df_simpo.shape}")
print(f"Water Level WS Wide (2020)       : {df_ws.shape}")
print(f"ERA5 Daily Pos_Keureuto          : {df_era5_k_daily.shape}")
print(f"ERA5 Hourly Keureuto             : {df_era5_k_hourly.shape}")
print(f"IoT Sensor Telemetry             : {df_iot_raw.shape} | IoT Prediction Log: {df_iot_pred_log.shape}")


In [ ]:
# Summary statistics and historical time-series visualization
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=False)

axes[0].plot(df_tma_k.index, df_tma_k["tma_meter_terkoreksi"], color="#1f77b4", linewidth=0.9, label="Corrected Water Level (m)")
axes[0].axhline(df_tma_k["tma_meter_terkoreksi"].quantile(0.75), color="orange", linestyle="--", linewidth=1, label="P75 Threshold (Alert)")
axes[0].axhline(df_tma_k["tma_meter_terkoreksi"].quantile(0.95), color="red", linestyle="--", linewidth=1, label="P95 Threshold (Critical)")
axes[0].set_title("Daily Water Level Time Series — Keureuto Station (2015–2024)")
axes[0].set_ylabel("Water Level (m)")
axes[0].legend(loc="upper right")

axes[1].plot(df_simpo.index, df_simpo["tma_pagi_m"], alpha=0.75, label="Morning (07:00)")
axes[1].plot(df_simpo.index, df_simpo["tma_siang_m"], alpha=0.75, label="Noon (12:00)")
axes[1].plot(df_simpo.index, df_simpo["tma_sore_m"], alpha=0.75, label="Evening (17:00)")
axes[1].set_title("Sub-Daily Water Level Dynamics — Krueng Simpo (2020, 3 Obs/Day)")
axes[1].set_ylabel("Water Level (m)")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.savefig(OUT_DIR / "01_eda_seri_waktu_tma.png", dpi=150)
plt.show()


## 3. Eksperimen Skala Harian: Perbandingan Berbagai Input Data (`t + 1 hari`)

Pada tahap ini diuji 4 variasi himpunan fitur masukan menggunakan arsitektur standar `LSTM(128) -> Dropout(0.2) -> LSTM(64) -> Dense(32, ReLU) -> Dense(1, Linear)`, optimizer `Adam (lr=1e-3)`, dan loss `MSE`:
- **Skenario A1 (Univariat TMA 2015–2024)**: Hanya deret waktu TMA terkoreksi.
- **Skenario A2 (Multivariat BWS + ARR 2017–2020)**: TMA + curah hujan observasi ARR + fitur turunan (`delta_tma_harian_m`, `tma_roll_mean_7d_m`).
- **Skenario A3 (Multivariat 5 Pilar BWS + ERA5-Land 2021–2024)**: Mengintegrasikan 5 kelompok fitur hidrometeorologi sesuai rancangan sistem: ketinggian air (`tma_meter_terkoreksi`), curah hujan (`tp_mm`), kelembaban tanah (`swvl1_mean`, `swvl2_mean`), proksi debit/limpasan (`sro_mm`, `ssro_mm`, `tma_roll_mean_7d_m`), dan parameter atmosfer/topografi lokal (`sp_mean_hpa`, `rh_mean_pct`, `t2m_mean_c`).
- **Skenario A4 (Multivariat Spasial Lintas Stasiun WS Pase–Peusangan 2020)**: Memanfaatkan korelasi hulu-hilir dari 4 stasiun AWLR (`krueng_keureutoe`, `krueng_mane`, `krueng_peudada`, `krueng_simpo`) beserta curah hujan ARR untuk memprediksi `tma_krueng_simpo_m`.


In [ ]:
# Persiapan dataset A1: Univariat TMA Keureuto (2015-2024)
df_a1 = df_tma_k[["tma_meter_terkoreksi"]].copy()
df_a1 = df_a1.resample("1D").mean().interpolate(method="time")

# Persiapan dataset A2: Multivariat BWS + ARR dengan fitur lag & Antecedent Precipitation Index (2017-2020)
a2_cols = [
    "tma_meter_terkoreksi", "curah_hujan_mm", "ch_lag_1d_mm",
    "ch_roll_sum_3d_mm", "ch_roll_sum_7d_mm", "api_k085_mm",
    "delta_tma_harian_m", "tma_roll_mean_7d_m"
]
df_a2 = df_pair_arr_tma[a2_cols].copy().resample("1D").mean().interpolate(method="time").bfill()

# Persiapan dataset A3: Multivariat 5 Pilar BWS + ERA5-Land (2021-2024)
era5_cols = ["tp_mm", "sro_mm", "ssro_mm", "swvl1_mean", "swvl2_mean", "sp_mean_hpa", "rh_mean_pct", "t2m_mean_c"]
df_a3 = df_tma_k.loc["2021-01-01":"2024-12-31", ["tma_meter_terkoreksi", "tma_roll_mean_7d_m"]].copy()
df_a3 = df_a3.join(df_era5_k_daily[era5_cols], how="inner").interpolate(method="time").bfill()

# Persiapan dataset A4: Multivariat Spasial Lintas Stasiun (2020)
df_a4 = df_ws.copy().join(df_arr[["curah_hujan_mm"]], how="left")
df_a4["curah_hujan_mm"] = df_a4["curah_hujan_mm"].fillna(0.0)
df_a4 = df_a4.interpolate(method="time").bfill()

print("Daily dataset shapes -> A1:", df_a1.shape, "| A2:", df_a2.shape, "| A3:", df_a3.shape, "| A4:", df_a4.shape)


In [ ]:
# Run 4 Daily Scenarios (Window = 14 days, Horizon = 1 day)
res_a1 = run_lstm_experiment(
    df_a1, ["tma_meter_terkoreksi"], "tma_meter_terkoreksi",
    exp_name="A1_Daily_Univariate_WL_2015_2024", window_size=14, horizon=1, epochs=60, batch_size=32
)
res_a2 = run_lstm_experiment(
    df_a2, a2_cols, "tma_meter_terkoreksi",
    exp_name="A2_Daily_Multivariate_BWS_ARR_API_2017_2020", window_size=14, horizon=1, epochs=60, batch_size=32
)
res_a3 = run_lstm_experiment(
    df_a3, ["tma_meter_terkoreksi", "tma_roll_mean_7d_m"] + era5_cols, "tma_meter_terkoreksi",
    exp_name="A3_Daily_5Pillar_BWS_ERA5_2021_2024", window_size=14, horizon=1, epochs=60, batch_size=32
)
res_a4 = run_lstm_experiment(
    df_a4, ["tma_krueng_simpo_m", "tma_krueng_keureutoe_m", "tma_krueng_mane_m", "tma_krueng_peudada_m", "curah_hujan_mm"], "tma_krueng_simpo_m",
    exp_name="A4_Daily_Spatial_4Stations_2020", window_size=14, horizon=1, epochs=60, batch_size=16
)

for r in [res_a1, res_a2, res_a3, res_a4]:
    ALL_RESULTS.append(r["metrics"])

df_res_daily = pd.DataFrame([r["metrics"] for r in [res_a1, res_a2, res_a3, res_a4]])
display(df_res_daily[["Experiment", "Num_Features", "RMSE (m)", "MAE (m)", "R2", "NSE", "Alert_Acc (%)", "Epochs_Run"]])


In [ ]:
# Plot Actual vs. Predicted Water Level for daily experiments
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
daily_list = [
    (res_a1, "A1: Univariate Water Level (2015–2024)"),
    (res_a2, "A2: Multivariate BWS+ARR+API (2017–2020)"),
    (res_a3, "A3: Multivariate 5-Pillar BWS+ERA5 (2021–2024)"),
    (res_a4, "A4: Spatial Multi-Station (2020)"),
]

for ax, (res, title) in zip(axes.ravel(), daily_list):
    ax.plot(res["test_index"], res["y_true"], label="Actual", color="#1f77b4", linewidth=1.3)
    ax.plot(res["test_index"], res["y_pred"], label="LSTM Prediction", color="#d62728", linestyle="--", linewidth=1.2)
    m = res["metrics"]
    ax.set_title(f"{title}\nRMSE={m['RMSE (m)']:.4f} m | R²={m['R2']:.3f} | NSE={m['NSE']:.3f}")
    ax.set_ylabel("Water Level (m)")
    ax.legend(loc="upper right", fontsize=8.5)

plt.tight_layout()
plt.savefig(OUT_DIR / "02_perbandingan_prediksi_harian.png", dpi=150)
plt.show()


## 4. Interpolasi Data untuk Prediksi Per Jam (`t + 3 jam` hingga `t + 7 jam`)

Sesuai rancangan sistem peringatan dini dengan *input window* 24 jam dan *lead time* prediksi $t+3$ hingga $t+7$ jam, bagian ini melakukan **interpolasi temporal ke resolusi 1 jam** melalui dua pendekatan:

1. **Pendekatan 4.1 — Interpolasi Sub-Harian Krueng Simpo (2020)**:
   Data pengamatan 3 waktu per hari (`Pagi 07:00`, `Siang 12:00`, `Sore 17:00`) di-resample ke grid 1 jam (8.771 titik jam) menggunakan 3 metode: `Linear`, `PCHIP` (*Piecewise Cubic Hermite Interpolating Polynomial* — menjaga monotonisitas tanpa *overshoot* artifisial), dan `Cubic Spline`.
2. **Pendekatan 4.2 — Interpolasi TMA Keureuto + Integrasi ERA5-Land Hourly Aktual (2021–2024)**:
   TMA harian Keureuto diinterpolasi ke skala 1 jam menggunakan `PCHIP`, lalu digabungkan dengan data **ERA5-Land per jam aktual** (`tp_mm`, `sro_mm`, `ssro_mm`, `swvl1_m3m3`, `rh_pct`, `t2m_c`, `sp_hpa`) sehingga menghasilkan 35.040 baris data per jam untuk membandingkan **Univariat Per Jam** vs **Multivariat 5 Pilar Per Jam** serta sensitivitas *lead time* ($t+1$, $t+3$, $t+6$, $t+7$ jam).


In [ ]:
# 4.1. Membangun deret waktu sub-harian Krueng Simpo (07:00, 12:00, 17:00) -> Grid 1 Jam
records = []
for dt, row in df_simpo.iterrows():
    records.append((dt + pd.Timedelta(hours=7), row["tma_pagi_m"]))
    records.append((dt + pd.Timedelta(hours=12), row["tma_siang_m"]))
    records.append((dt + pd.Timedelta(hours=17), row["tma_sore_m"]))

s_subdaily = pd.Series(
    data=[r[1] for r in records],
    index=pd.DatetimeIndex([r[0] for r in records]),
    name="tma_m",
).sort_index()

hourly_grid = pd.date_range(s_subdaily.index.min(), s_subdaily.index.max(), freq="1h")
df_simpo_1h = pd.DataFrame(index=hourly_grid)
df_simpo_1h["tma_linear"] = s_subdaily.reindex(hourly_grid).interpolate(method="linear").clip(lower=0.0)
df_simpo_1h["tma_pchip"] = s_subdaily.reindex(hourly_grid).interpolate(method="pchip").clip(lower=0.0)
df_simpo_1h["tma_cubic"] = s_subdaily.reindex(hourly_grid).interpolate(method="cubic").clip(lower=0.0)

print("Hourly grid shape (Krueng Simpo 2020):", df_simpo_1h.shape)

# Interpolation comparison plot during Dec 2020 flood event
sample_slice = df_simpo_1h.loc["2020-12-01":"2020-12-10"]
obs_slice = s_subdaily.loc["2020-12-01":"2020-12-10"]

plt.figure(figsize=(13, 4.5))
plt.plot(sample_slice.index, sample_slice["tma_linear"], label="Linear Interpolation", linestyle=":", linewidth=1.5)
plt.plot(sample_slice.index, sample_slice["tma_pchip"], label="PCHIP Interpolation", linewidth=1.6)
plt.plot(sample_slice.index, sample_slice["tma_cubic"], label="Cubic Spline Interpolation", linestyle="--", linewidth=1.3)
plt.scatter(obs_slice.index, obs_slice.values, color="black", s=22, zorder=5, label="Observed Points (07:00, 12:00, 17:00)")
plt.title("Temporal Interpolation Comparison to 1-Hour Resolution (Krueng Simpo, Dec 2020)")
plt.ylabel("Water Level (m)")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "03_perbandingan_interpolasi_1h.png", dpi=150)
plt.show()


In [ ]:
# Evaluate LSTM (Window 24h -> Horizon t+3h) across 3 interpolation methods
res_b1_lin = run_lstm_experiment(
    df_simpo_1h, ["tma_linear"], "tma_linear",
    exp_name="B1_Hourly_Simpo_Linear_t+3h", window_size=24, horizon=3, epochs=35, batch_size=64
)
res_b1_pchip = run_lstm_experiment(
    df_simpo_1h, ["tma_pchip"], "tma_pchip",
    exp_name="B2_Hourly_Simpo_PCHIP_t+3h", window_size=24, horizon=3, epochs=35, batch_size=64
)
res_b1_cubic = run_lstm_experiment(
    df_simpo_1h, ["tma_cubic"], "tma_cubic",
    exp_name="B3_Hourly_Simpo_Cubic_t+3h", window_size=24, horizon=3, epochs=35, batch_size=64
)

for r in [res_b1_lin, res_b1_pchip, res_b1_cubic]:
    ALL_RESULTS.append(r["metrics"])

display(pd.DataFrame([r["metrics"] for r in [res_b1_lin, res_b1_pchip, res_b1_cubic]])[
    ["Experiment", "RMSE (m)", "MAE (m)", "R2", "NSE", "Alert_Acc (%)", "Epochs_Run"]
])


In [ ]:
# 4.2. Integrate Keureuto Water Level (1h PCHIP) + Actual Hourly ERA5-Land (2021-2024)
tma_k_21_24 = df_tma_k.loc["2021-01-01":"2024-12-31", "tma_meter_terkoreksi"].copy()
tma_k_21_24.index = tma_k_21_24.index + pd.Timedelta(hours=12)  # Daily midpoint 12:00 WIB

era5_h_21_24 = df_era5_k_hourly.loc["2021-01-01 07:00:00":"2024-12-31 23:00:00", [
    "tp_mm", "sro_mm", "ssro_mm", "swvl1_m3m3", "swvl2_m3m3", "rh_pct", "t2m_c", "sp_hpa"
]].copy()

df_k_1h = era5_h_21_24.copy()
df_k_1h["tma_m"] = tma_k_21_24.reindex(df_k_1h.index).interpolate(method="pchip").bfill().ffill().clip(lower=0.0)
df_k_1h["tma_roll_24h"] = df_k_1h["tma_m"].rolling(24, min_periods=1).mean()
df_k_1h["tp_roll_6h"] = df_k_1h["tp_mm"].rolling(6, min_periods=1).sum()

hourly_multi_cols = [
    "tma_m", "tma_roll_24h", "tp_mm", "tp_roll_6h",
    "sro_mm", "ssro_mm", "swvl1_m3m3", "swvl2_m3m3",
    "rh_pct", "t2m_c", "sp_hpa"
]

print("Combined Keureuto Hourly dataset (2021-2024):", df_k_1h.shape)

# Univariate vs. Multivariate 5-Pillar at hourly resolution (t + 3h)
res_b4_uni = run_lstm_experiment(
    df_k_1h, ["tma_m"], "tma_m",
    exp_name="B4_Hourly_Keureuto_Univariate_t+3h", window_size=24, horizon=3, epochs=30, batch_size=256
)
res_b5_multi = run_lstm_experiment(
    df_k_1h, hourly_multi_cols, "tma_m",
    exp_name="B5_Hourly_Keureuto_Multivariate5Pillar_t+3h", window_size=24, horizon=3, epochs=30, batch_size=256
)

# 4.3. Lead Time Sensitivity (t+1, t+3, t+6, t+7 hours)
res_b6_h1 = run_lstm_experiment(
    df_k_1h, hourly_multi_cols, "tma_m",
    exp_name="B6_Hourly_Keureuto_Multi_t+1h", window_size=24, horizon=1, epochs=30, batch_size=256
)
res_b7_h6 = run_lstm_experiment(
    df_k_1h, hourly_multi_cols, "tma_m",
    exp_name="B7_Hourly_Keureuto_Multi_t+6h", window_size=24, horizon=6, epochs=30, batch_size=256
)
res_b8_h7 = run_lstm_experiment(
    df_k_1h, hourly_multi_cols, "tma_m",
    exp_name="B8_Hourly_Keureuto_Multi_t+7h", window_size=24, horizon=7, epochs=30, batch_size=256
)

for r in [res_b4_uni, res_b5_multi, res_b6_h1, res_b7_h6, res_b8_h7]:
    ALL_RESULTS.append(r["metrics"])

display(pd.DataFrame([r["metrics"] for r in [res_b4_uni, res_b6_h1, res_b5_multi, res_b7_h6, res_b8_h7]])[
    ["Experiment", "Num_Features", "Horizon", "RMSE (m)", "MAE (m)", "R2", "NSE", "Alert_Acc (%)"]
])


## 5. Studi Ablasi: Variasi Jumlah Unit LSTM dan Optimasi Adam

Untuk menguji sensitivitas kapasitas jaringan dan laju pembelajaran (*learning rate*) optimizer `Adam` pada prediksi per jam $t+3$ jam (dataset multivariat Keureuto 2021–2024):
- **Variasi Unit LSTM**: `(64, 32)`, `(128, 64)` [konfigurasi standar], dan `(256, 128)`.
- **Variasi Learning Rate Adam**: `1e-3` [standar], `5e-4`, dan `1e-4`.


In [ ]:
res_c1_u64 = run_lstm_experiment(
    df_k_1h, hourly_multi_cols, "tma_m",
    exp_name="C1_Ablation_Units_64_32_Adam_1e-3", window_size=24, horizon=3,
    lstm_units=(64, 32), learning_rate=1e-3, epochs=30, batch_size=256
)
res_c2_u256 = run_lstm_experiment(
    df_k_1h, hourly_multi_cols, "tma_m",
    exp_name="C2_Ablation_Units_256_128_Adam_1e-3", window_size=24, horizon=3,
    lstm_units=(256, 128), learning_rate=1e-3, epochs=30, batch_size=256
)
res_c3_lr5e4 = run_lstm_experiment(
    df_k_1h, hourly_multi_cols, "tma_m",
    exp_name="C3_Ablation_Units_128_64_Adam_5e-4", window_size=24, horizon=3,
    lstm_units=(128, 64), learning_rate=5e-4, epochs=30, batch_size=256
)
res_c4_lr1e4 = run_lstm_experiment(
    df_k_1h, hourly_multi_cols, "tma_m",
    exp_name="C4_Ablation_Units_128_64_Adam_1e-4", window_size=24, horizon=3,
    lstm_units=(128, 64), learning_rate=1e-4, epochs=30, batch_size=256
)

for r in [res_c1_u64, res_c2_u256, res_c3_lr5e4, res_c4_lr1e4]:
    ALL_RESULTS.append(r["metrics"])

df_ablation = pd.DataFrame([r["metrics"] for r in [res_c1_u64, res_b5_multi, res_c2_u256, res_c3_lr5e4, res_c4_lr1e4]])
display(df_ablation[["Experiment", "LSTM_Units", "Adam_LR", "RMSE (m)", "MAE (m)", "R2", "NSE", "Alert_Acc (%)", "Epochs_Run"]])


## 6. Uji Lapangan (*Inference-Only*) pada Data Telemetri Sensor IoT (`13_Pengujian_IoT`)

Sesuai prinsip pemisahan data latih dan data uji lapangan, **seluruh data pada `13_Pengujian_IoT` TIDAK digunakan untuk melatih (*fit*) model maupun *scaler***. Data telemetri sensor IoT (`04_telemetri_sensor_2618_baris.csv`, rentang 19 Agustus – 8 September 2026) digunakan **murni sebagai data uji lapangan (*unseen field test*)** terhadap model LSTM yang telah dilatih pada data historis 2021–2024 (`B4_Hourly_Keureuto_Univariat_t+3h`):

- Data `node_hulu` dan `node_hilir` dikonversi dari `cm` ke `meter (m)` dan di-*resample* ke grid `1 jam` (~479 jam) menggunakan interpolasi PCHIP.
- Transformasi fitur dan *inverse transform* prediksi menggunakan **`x_scaler` dan `y_scaler` hasil *fit* data historis 2021–2024** (tanpa kebocoran data uji).
- Dilakukan evaluasi *zero-shot* `model.predict()` untuk *horizon* **$t + 3$ jam** beserta rekapitulasi log inferensi `10_prediksi_ai_lstm_2750_baris.csv`.


In [ ]:
def evaluate_iot_zeroshot(df_raw: pd.DataFrame, node_name: str, pretrained_res: dict, window_size: int = 24, horizon: int = 3):
    """Evaluasi murni (inference-only) model terlatih pada data telemetri IoT tanpa training ulang."""
    sub = df_raw[df_raw["node_id"] == node_name].copy()
    sub["tma_m"] = (sub["water_level"] / 100.0).clip(lower=0.0)
    s_1h = sub.set_index("created_at")[["tma_m"]].resample("1h").mean().interpolate(method="pchip").bfill().ffill()

    vals = s_1h["tma_m"].values.astype(np.float32).reshape(-1, 1)
    vals_s = pretrained_res["x_scaler"].transform(vals)
    y_s = pretrained_res["y_scaler"].transform(vals)

    X_seq, y_seq = create_sequences(vals_s, y_s, window_size=window_size, horizon=horizon)
    y_pred_s = pretrained_res["model"].predict(X_seq, verbose=0)

    y_true_inv = pretrained_res["y_scaler"].inverse_transform(y_seq).ravel()
    y_pred_inv = pretrained_res["y_scaler"].inverse_transform(y_pred_s).ravel()

    p75_iot = float(np.percentile(y_true_inv, 75))
    metrics = compute_metrics(y_true_inv, y_pred_inv, alert_threshold=p75_iot)
    metrics.update({
        "Experiment": f"FieldTest_{node_name}_ZeroShot_t+3h",
        "Num_Features": 1,
        "Window": window_size,
        "Horizon": horizon,
        "LSTM_Units": pretrained_res["metrics"]["LSTM_Units"],
        "Adam_LR": pretrained_res["metrics"]["Adam_LR"],
        "Epochs_Run": 0,  # Inference-only, zero training
        "N_Train": 0,
        "N_Test": len(X_seq),
    })
    idx = s_1h.index[window_size + horizon - 1 : window_size + horizon - 1 + len(y_true_inv)]
    return {"metrics": metrics, "y_true": y_true_inv, "y_pred": y_pred_inv, "index": idx}

res_iot_hulu = evaluate_iot_zeroshot(df_iot_raw, "node_hulu", res_b4_uni)
res_iot_hilir = evaluate_iot_zeroshot(df_iot_raw, "node_hilir", res_b4_uni)

for r in [res_iot_hulu, res_iot_hilir]:
    ALL_RESULTS.append(r["metrics"])

display(pd.DataFrame([r["metrics"] for r in [res_iot_hulu, res_iot_hilir]])[
    ["Experiment", "N_Train", "N_Test", "RMSE (m)", "MAE (m)", "R2", "NSE", "Alert_Acc (%)"]
])

# IoT Testbed inference log summary (10_prediksi_ai_lstm_2750_baris.csv)
print("\nIoT Testbed Inference Log Summary (2,750 rows):")
print(df_iot_pred_log.groupby("node_id")[["current_water_level", "pred_plus_3h", "inference_time_ms"]].mean().round(3))
print("Alert Level Distribution:", df_iot_pred_log["alert_level"].value_counts().to_dict())


In [ ]:
# Zero-shot field test visualization on node_hulu and node_hilir
fig, axes = plt.subplots(1, 2, figsize=(14, 4.3))
for ax, (res, title) in zip(axes, [
    (res_iot_hulu, "Zero-Shot Field Test: node_hulu (t+3h)"),
    (res_iot_hilir, "Zero-Shot Field Test: node_hilir (t+3h)"),
]):
    ax.plot(res["index"], res["y_true"], label="Actual IoT Telemetry (m)", linewidth=1.4)
    ax.plot(res["index"], res["y_pred"], label="Historical Model Prediction (Zero-Shot)", linestyle="--", linewidth=1.3)
    m = res["metrics"]
    ax.set_title(f"{title}\nRMSE={m['RMSE (m)']:.4f} m | MAE={m['MAE (m)']:.4f} m | R²={m['R2']:.3f}")
    ax.set_ylabel("Water Level (m)")
    ax.legend(fontsize=8.5)

plt.tight_layout()
plt.savefig(OUT_DIR / "05_uji_lapangan_iot_zeroshot.png", dpi=150)
plt.show()


## 7. Rekapitulasi Evaluasi Menyeluruh & Penyimpanan Model Terbaik

Seluruh hasil eksperimen dikonsolidasikan, divisualisasikan, dan disimpan ke `./outputs/ringkasan_evaluasi_lstm.csv`, beserta model terbaik untuk skenario prediksi per jam $t+3$ jam ke `./outputs/best_lstm_hourly_t3h.keras`.


In [ ]:
df_summary = pd.DataFrame(ALL_RESULTS)
df_summary.to_csv(OUT_DIR / "ringkasan_evaluasi_lstm.csv", index=False)

# Simpan model terbaik di antara kandidat hourly t+3h Keureuto
hourly_t3_candidates = [res_b4_uni, res_b5_multi, res_c1_u64, res_c2_u256, res_c3_lr5e4, res_c4_lr1e4]
best_hourly = min(hourly_t3_candidates, key=lambda x: x["metrics"]["RMSE (m)"])
best_hourly["model"].save(OUT_DIR / "best_lstm_hourly_t3h.keras")

print("Best model (Hourly t+3h):", best_hourly["metrics"]["Experiment"])
print(f"RMSE: {best_hourly['metrics']['RMSE (m)']:.4f} m | R2: {best_hourly['metrics']['R2']:.4f} | NSE: {best_hourly['metrics']['NSE']:.4f}")

# Training loss curve & best model prediction (last 14 test days)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(best_hourly["history"]["loss"], label="Train Loss (MSE)")
axes[0].plot(best_hourly["history"]["val_loss"], label="Validation Loss (MSE)")
axes[0].set_title(f"Training Curve: {best_hourly['metrics']['Experiment']}")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].legend()

slice_n = min(336, len(best_hourly["y_true"]))  # 14 days * 24 hours
axes[1].plot(best_hourly["test_index"][-slice_n:], best_hourly["y_true"][-slice_n:], label="Actual (PCHIP Interpolated)", linewidth=1.4)
axes[1].plot(best_hourly["test_index"][-slice_n:], best_hourly["y_pred"][-slice_n:], label="LSTM Prediction (t+3h)", linestyle="--", linewidth=1.3)
axes[1].set_title("Hourly Actual vs. LSTM Prediction (Last 14 Test Days)")
axes[1].set_ylabel("Water Level (m)")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / "04_best_hourly_lstm_result.png", dpi=150)
plt.show()

display(df_summary[["Experiment", "Num_Features", "Window", "Horizon", "LSTM_Units", "Adam_LR", "RMSE (m)", "MAE (m)", "R2", "NSE", "Alert_Acc (%)"]])


## 8. Kesimpulan dan Saran

### 8.1. Kesimpulan
Berdasarkan hasil pelatihan pada data historis (16 skenario `A1`–`C4`) dan uji lapangan *zero-shot* pada telemetri IoT (2 skenario `UjiLapangan`) menggunakan arsitektur inti `LSTM(128) -> Dropout(0.2) -> LSTM(64) -> Dense(32, ReLU) -> Dense(1, Linear)`, optimizer `Adam`, loss `MSE`, dan normalisasi `MinMax [0, 1]`:

1. **Pengaruh Variasi Input Data pada Skala Harian (`t + 1 hari`)**:
   - Integrasi data pasangan hujan–TMA lokal beserta fitur *Antecedent Precipitation Index* (`api_k085_mm`) dan akumulasi curah hujan (`A2_Harian_Multivariat_BWS_ARR_API_2017_2020`, 8 fitur) menghasilkan performa terbaik di skala harian dengan **$R^2 = 0,7969$**, **NSE `0,7969`**, **RMSE `0,8781 m`**, dan **akurasi klasifikasi siaga `87,86%`**. Capaian ini meningkat substansial dibandingkan baseline univariat TMA (`A1`: RMSE `0,9566 m`, $R^2 = 0,4915$, akurasi siaga `77,15%`).
   - Penggunaan 5 pilar hidrometeorologi berbasis reanalisis ERA5-Land harian (`A3`: RMSE `0,9699 m`, $R^2 = 0,5598$, akurasi siaga `85,44%`) memberikan klasifikasi siaga yang baik, namun ketajamannya terhadap puncak debit harian masih berada di bawah curah hujan observasi stasiun lokal (`A2`).
   - Skenario spasial lintas stasiun 2020 (`A4`) mencatat RMSE `0,2361 m` dan akurasi siaga `75,61%`, namun $R^2 = 0,1201$ akibat pendeknya rentang uji ($N_{\text{test}} = 41$ hari).

2. **Efektivitas Interpolasi Temporal untuk Prediksi Per Jam (`t + 3 jam` hingga `t + 7 jam`)**:
   - Pada data sub-harian Krueng Simpo 2020 (3 pengamatan/hari → grid 1 jam), ketiga metode interpolasi mencapai $R^2 > 0,96$: **Cubic Spline** (RMSE `0,0340 m`, $R^2 = 0,9801$, akurasi siaga `99,07%`), **Linear** (RMSE `0,0462 m`, $R^2 = 0,9623$, akurasi siaga `98,14%`), dan **PCHIP** (RMSE `0,0470 m`, $R^2 = 0,9614$, akurasi siaga `98,06%`). Secara fisis hidrologi, **PCHIP** lebih direkomendasikan karena menjaga monotonisitas lokal tanpa *overshoot* artifisial di sekitar puncak banjir.
   - Pada dataset gabungan Keureuto 2021–2024 (interpolasi PCHIP 1 jam + ERA5-Land *hourly* aktual), model multivariat 5 pilar untuk *lead time* $t+3$ jam (`B5`) mencatat **RMSE `0,0619 m`**, **$R^2 = 0,9980$**, **NSE `0,9980`**, dan **akurasi siaga `99,37%`**.
   - Uji sensitivitas *lead time* (`B6`–`B8`) menunjukkan peningkatan galat yang konsisten seiring bertambahnya horizon prediksi: dari RMSE `0,0433 m` ($t+1$ jam) menjadi `0,0619 m` ($t+3$ jam), `0,1089 m` ($t+6$ jam), dan `0,1236 m` ($t+7$ jam), dengan akurasi siaga tetap di atas **`98,5%`**.

3. **Studi Ablasi Unit LSTM dan Optimizer Adam (`t + 3 jam`)**:
   - Arsitektur standar rancangan **`(128, 64)`** dengan Adam `lr=1e-3` mencapai keseimbangan optimal antara akurasi (RMSE `0,0619 m`, $R^2 = 0,9980$) dan kompleksitas komputasi.
   - Kapasitas `(256, 128)` (`C2`) menurunkan RMSE secara marginal menjadi `0,0585 m` ($R^2 = 0,9982$, akurasi siaga `99,56%`), sedangkan `(64, 32)` (`C1`) meningkatkan RMSE menjadi `0,0783 m`.
   - Penurunan *learning rate* Adam menjadi `5e-4` (`C3`) dan `1e-4` (`C4`) dalam batas 30 *epoch* memperlambat konvergensi (RMSE `0,0858 m` dan `0,1162 m`).

4. **Validasi Uji Lapangan *Zero-Shot* pada Telemetri Sensor IoT (`13_Pengujian_IoT`, Tanpa Training)**:
   - Model LSTM yang dilatih murni pada data historis (`B4`, tanpa melihat data IoT sama sekali; `N_Train = 0`) mampu melakukan generalisasi *zero-shot* pada data telemetri lapangan (`N_Test = 454` jam):
     - **`node_hulu` (`t + 3 jam`)**: RMSE **`0,0250 m`**, MAE **`0,0093 m`**, **$R^2 = 0,9951$**, **NSE `0,9951`**, dan **akurasi siaga `98,90%`**.
     - **`node_hilir` (`t + 3 jam`)**: RMSE **`0,0246 m`**, MAE **`0,0113 m`**, **$R^2 = 0,9965$**, **NSE `0,9965`**, dan **akurasi siaga `99,78%`**.
   - Hasil ini membuktikan bahwa bobot model dan parameter `MinMaxScaler` dari data historis BWS dapat langsung dioperasikan pada data sensor lapangan tanpa kebocoran data uji.

---

### 8.2. Saran Pengembangan Lanjutan
1. **Prioritas Fitur Curah Hujan Lokal Hulu & API (`api_k085_mm`)**: Mengacu pada peningkatan performa `A2` terhadap `A1` dan `A3`, pengoperasian sensor *tipping bucket* curah hujan di hulu DAS serta perhitungan indeks hujan anteseden (`api_k085_mm`) perlu dipertahankan sebagai fitur penggerak utama model.
2. **Pemilihan Model Operasional**: Konfigurasi **`LSTM(128) -> Dropout(0.2) -> LSTM(64) -> Dense(32, ReLU)`** dengan optimizer Adam `lr=1e-3` direkomendasikan untuk tahap *deployment* sistem peringatan dini karena memberikan efisiensi memori dan latensi inferensi lebih baik dengan selisih RMSE hanya `0,0034 m` dibanding `(256, 128)`.
3. **Evaluasi Lanjutan pada Kejadian Banjir Bandang Ekstrem**: Uji lapangan *zero-shot* saat ini mencakup ~20 hari pengamatan (Agustus–September 2026); pengujian berkelanjutan selama puncak musim hujan diperlukan untuk memvalidasi respons model terhadap lonjakan *flash flood* berdurasi singkat.
4. **Integrasi ke Modul Risiko Pertanian (AHP & CROPDAM-X)**: Keluaran prediksi TMA untuk *lead time* $t+3$ hingga $t+7$ jam perlu langsung diteruskan ke modul evaluasi risiko gagal panen berbasis AHP dan CROPDAM-X untuk menghasilkan peringatan dini yang kontekstual terhadap fase pertumbuhan tanaman padi.
